# Typed instrument API reference catalog

**Reference note:** This catalog complements the asset-class notebooks; start with `02_pricing/pricing_fundamentals.ipynb` and the relevant instrument deep dive before using it as an API checklist.

**Purpose:** Learn the typed instrument pattern once, then use a compact external catalog to verify every priceable instrument class exposed through `finstack_quant.valuations.instruments.*`.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb` and at least one asset-class deep dive under `02_pricing/instruments/`.

**What you'll learn:**

- Build a typed instrument with `Cls.from_json(json_str)`.
- Validate and serialize with `validate()` and `to_json()`.
- Price typed instruments against a shared `MarketContext`.
- Use the external catalog as a reference, not as the main teaching narrative.

Every instrument under `finstack_quant.valuations.instruments.*` shares one surface: `from_json`, `validate`, `to_json`, `price`, and `price_with_metrics`. The full 70-instrument catalog is stored in `data/typed_instrument_catalog.json` so the notebook stays short while the runnable coverage remains complete.


## The typed pattern in three steps

The catalog is large, but the user-facing pattern is small. Pick a class, wrap its tagged JSON payload, validate it, and then use the same typed object for serialization or pricing.


In [ ]:
import json
from pathlib import Path

from finstack_quant.valuations.instruments.fixed_income import Bond
from finstack_quant.valuations.instruments.equity import EquityOption
from finstack_quant.valuations.instruments.rates import InterestRateSwap

catalog = json.loads(Path("data/typed_instrument_catalog.json").read_text())
examples = {
    "Bond": Bond,
    "EquityOption": EquityOption,
    "InterestRateSwap": InterestRateSwap,
}

for class_name, cls in examples.items():
    tagged_payload = catalog[next(module for module, specs in catalog.items() if class_name in specs)][class_name]
    instrument = cls.from_json(json.dumps(tagged_payload))
    instrument.validate()
    print(f"{class_name}: {type(instrument).__name__}, json bytes={len(instrument.to_json())}")


## Instrument catalog: `from_json` -> `validate` -> `to_json`

`CATALOG` maps each module to `{ClassName: inner_spec}`. We import every class, rebuild it from its spec, validate, and confirm the JSON round-trips.

In [1]:
import json
import importlib
from pathlib import Path

# The full verified catalog lives outside the notebook so the lesson stays readable.
CATALOG = json.loads(Path("data/typed_instrument_catalog.json").read_text())

# Keep the class names explicit in code for API-coverage audits and quick review.
COVERED_INSTRUMENT_CLASSES = [
    "CommodityAsianOption",
    "CommodityForward",
    "CommodityOption",
    "CommoditySpreadOption",
    "CommoditySwap",
    "CommoditySwaption",
    "CDSIndex",
    "CDSOption",
    "CDSTranche",
    "CreditDefaultSwap",
    "Autocallable",
    "CliquetOption",
    "DiscountedCashFlow",
    "Equity",
    "EquityIndexFuture",
    "EquityOption",
    "EquityTotalReturnSwap",
    "LeveredRealEstateEquity",
    "PrivateMarketsFund",
    "RealEstateAsset",
    "VarianceSwap",
    "VolatilityIndexFuture",
    "VolatilityIndexOption",
    "AsianOption",
    "BarrierOption",
    "Basket",
    "LookbackOption",
    "AgencyCmo",
    "AgencyMbsPassthrough",
    "AgencyTba",
    "Bond",
    "BondFuture",
    "ConvertibleBond",
    "DollarRoll",
    "FIIndexTotalReturnSwap",
    "InflationLinkedBond",
    "RevolvingCredit",
    "StructuredCredit",
    "TermLoan",
    "FxBarrierOption",
    "FxDigitalOption",
    "FxForward",
    "FxOption",
    "FxSpot",
    "FxSwap",
    "FxTouchOption",
    "FxVarianceSwap",
    "Ndf",
    "QuantoOption",
    "BasisSwap",
    "BermudanSwaption",
    "CallableRangeAccrual",
    "CapFloor",
    "CmsOption",
    "CmsSpreadOption",
    "CmsSwap",
    "Deposit",
    "ForwardRateAgreement",
    "InflationCapFloor",
    "InflationSwap",
    "InterestRateFuture",
    "InterestRateSwap",
    "IrFutureOption",
    "RangeAccrual",
    "Repo",
    "Snowball",
    "Swaption",
    "Tarn",
    "XccySwap",
    "YoYInflationSwap"
]
ok = 0
for module_name, instruments in CATALOG.items():
    mod = importlib.import_module(f"finstack_quant.valuations.instruments.{module_name}")
    for class_name, spec in instruments.items():
        cls = getattr(mod, class_name)
        inst = cls.from_json(json.dumps(spec))   # build typed instrument
        inst.validate()                          # structural validation
        assert len(inst.to_json()) > 0            # canonical serialization
        ok += 1
    print(f"{module_name:18} {len(instruments):2d} instrument classes validated")
print(f"\nTotal: {ok} typed instrument classes built + validated + serialized")


commodity           6 instrument classes validated
credit_derivatives  4 instrument classes validated
equity             13 instrument classes validated
exotics             4 instrument classes validated
fixed_income       12 instrument classes validated
fx                 10 instrument classes validated
rates              21 instrument classes validated

Total: 70 typed instrument classes built + validated + serialized


## End-to-end pricing

The pricing pass below derives the market-data IDs referenced by `CATALOG`, builds one synthetic `MarketContext` from typed market-data classes, and then prices every instrument independently. Residual failures are annotated as example scaffolding gaps rather than raised, so the notebook remains runnable while still showing the full typed pricing surface.

In [2]:
from collections import defaultdict

MARKET_ID_MAP = defaultdict(set)
OPERATIONAL_REFERENCE_IDS = defaultdict(set)


def _walk_market_references(value, path=()):
    if isinstance(value, dict):
        for key, item in value.items():
            if isinstance(item, str) and (key.endswith("_id") or key in {"index_id", "ticker"}):
                yield key, item, path + (key,)
            yield from _walk_market_references(item, path + (key,))
    elif isinstance(value, list):
        for idx, item in enumerate(value):
            yield from _walk_market_references(item, path + (str(idx),))


def _market_kind(module_name, field, value):
    if field in {"discount_curve_id", "domestic_discount_curve_id", "foreign_discount_curve_id"}:
        return "discount_curve"
    if field == "credit_curve_id" and module_name == "fixed_income" and value == "USD-CREDIT-BBB":
        # The convertible example labels this credit curve, but its registered pricer requests a discount-style spread curve.
        return "discount_curve"
    if field in {"credit_curve_id", "credit_index_id"}:
        return "hazard_curve" if field == "credit_curve_id" else "credit_index"
    if field in {"inflation_index_id"}:
        return "inflation_curve"
    if field in {"vol_index_curve_id"} or value == "VIX":
        return "vol_index_curve"
    if field.endswith("vol_surface_id") or field in {"vol_surface_id", "fx_vol_id", "vol_of_vol_surface_id"}:
        return "vol_surface"
    if field in {"forward_curve_id", "floating_index_id", "leg1_forward_curve_id", "leg2_forward_curve_id"}:
        return "price_curve" if module_name == "commodity" or value.endswith("SPOT-AVG") else "forward_curve"
    if field == "index_id":
        if value.startswith("SOFR") or value.startswith("USD-") or value.startswith("EUR-"):
            return "forward_curve"
        return "scalar_price"
    if field in {"spot_id", "price_id", "market_value_id", "fx_spot_id", "fx_rate_id", "yield_id", "duration_id", "convexity_id", "bond_id", "ctd_bond_id"}:
        return "scalar_price"
    if field == "div_yield_id":
        return "dividend_yield"
    return "operational_reference"


for module_name, instruments in CATALOG.items():
    for class_name, payload in instruments.items():
        for field, value, _path in _walk_market_references(payload["spec"]):
            kind = _market_kind(module_name, field, value)
            if kind == "operational_reference":
                OPERATIONAL_REFERENCE_IDS[kind].add(value)
            else:
                MARKET_ID_MAP[kind].add(value)

# A few pricing engines request companion market objects that are implied by the spec rather than named directly.
MARKET_ID_MAP["discount_curve"].update({"EUR-OIS", "JPY-OIS", "USD-TREASURY"})
MARKET_ID_MAP["price_curve"].update({"CL-FORWARD", "WTI-FORWARD", "NG-FORWARD", "RBOB-FORWARD", "NG-SPOT-AVG"})
MARKET_ID_MAP["vol_surface"].update({"JPYUSD-VOL", "RBOB-VOL", "TECH-VOL", "USD-SWAPTION-VOL-2Y", "USD-SWAPTION-VOL-10Y"})
MARKET_ID_MAP["sabr_cube"].update({"USD-RFR-BVOL-CUBE-5Y", "USD-SWPNVOL", "USD-CMS10Y-VOL", "USD-SWAPTION-VOL-2Y", "USD-SWAPTION-VOL-10Y", "VIX-VOLVOL"})
MARKET_ID_MAP["scalar_price"].update({"CNYUSD-SPOT", "EURUSD-SPOT", "JPYUSD-SPOT", "TECH", "UST-10Y", "US91282CJL54"})

print("Derived market references:")
for kind in sorted(MARKET_ID_MAP):
    print(f"  {kind:16} {len(MARKET_ID_MAP[kind]):2d} ids")
print("Operational ids annotated if required:", len(OPERATIONAL_REFERENCE_IDS["operational_reference"]))


Derived market references:
  credit_index      1 ids
  discount_curve   10 ids
  dividend_yield    5 ids
  forward_curve     8 ids
  hazard_curve      2 ids
  inflation_curve   1 ids
  price_curve       5 ids
  sabr_cube         6 ids
  scalar_price     18 ids
  vol_index_curve   1 ids
  vol_surface      19 ids
Operational ids annotated if required: 23


In [3]:
from datetime import date, timedelta

from finstack_quant.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    FxMatrix,
    HazardCurve,
    InflationCurve,
    MarketContext,
    PriceCurve,
    VolCube,
    VolSurface,
    VolatilityIndexCurve,
)

AS_OF = date(2024, 1, 15)

DISCOUNT_KNOTS = [
    (0.0, 1.0000),
    (0.5, 0.9802),
    (1.0, 0.9608),
    (2.0, 0.9231),
    (5.0, 0.8187),
    (10.0, 0.6703),
    (30.0, 0.3010),
]
FORWARD_KNOTS = [
    (0.0, 0.0420),
    (0.5, 0.0430),
    (1.0, 0.0435),
    (2.0, 0.0440),
    (5.0, 0.0450),
    (10.0, 0.0460),
    (30.0, 0.0470),
]
PRICE_CURVE_LEVELS = {
    "CL-FORWARD": 76.0,
    "WTI-FORWARD": 74.0,
    "NG-FORWARD": 3.20,
    "RBOB-FORWARD": 2.40,
    "NG-SPOT-AVG": 3.20,
}
SCALAR_PRICES = {
    "AAPL-SPOT": 185.0,
    "CNYUSD-SPOT": 0.1400,
    "EQUITY-SPOT": 100.0,
    "EURUSD": 1.08,
    "EURUSD-SPOT": 1.08,
    "JPYUSD-SPOT": 0.0068,
    "NKY-SPOT": 39000.0,
    "SOFR-RATE": 0.045,
    "SPX-SPOT": 5200.0,
    "TECH": 42.0,
    "US-CORP-CONVEXITY": 0.02,
    "US-CORP-DURATION": 7.0,
    "US-CORP-INDEX": 100.0,
    "US-CORP-YIELD": 0.052,
    "US91282CJL54": 99.5,
    "UST-10Y": 99.5,
    "UST10Y-PRICE": 99.5,
    "UST_10Y_PRICE": 99.5,
    "VIX": 18.0,
}
DIVIDEND_YIELDS = {
    "AAPL-DIV": 0.006,
    "EQUITY-DIVYIELD": 0.010,
    "NKY-DIV": 0.012,
    "SPX-DIV": 0.015,
    "SPX-DIVYIELD": 0.015,
}
FORWARD_TENORS = {
    "EUR-EURIBOR-3M": 0.25,
    "SOFR-1M": 1.0 / 12.0,
    "SOFR-3M": 0.25,
    "USD-LIBOR-3M": 0.25,
    "USD-SOFR": 0.25,
    "USD-SOFR-1M": 1.0 / 12.0,
    "USD-SOFR-3M": 0.25,
    "USD-SOFR-6M": 0.50,
}


def _price_curve_knots(level):
    return [
        (0.0, level),
        (0.5, level * 1.01),
        (1.0, level * 1.02),
        (2.0, level * 1.03),
        (5.0, level * 1.05),
    ]


def _is_rate_vol_surface(surface_id):
    return any(token in surface_id for token in ("SWPN", "SWAPTION", "SOFR", "CMS", "RFR", "INFL"))


def _fixing_series(series_id, as_of):
    observations = []
    current = date(2023, 1, 1)
    end = as_of + timedelta(days=400)
    while current <= end:
        observations.append([current.isoformat(), 0.043])
        current += timedelta(days=1)
    return {
        "id": series_id,
        "currency": None,
        "observations": observations,
        "interpolation": "step",
    }


def build_shared_market_json(as_of=AS_OF):
    market = MarketContext()

    discount_ids = set(MARKET_ID_MAP["discount_curve"])
    forward_ids = set(MARKET_ID_MAP["forward_curve"]) - discount_ids - set(MARKET_ID_MAP["price_curve"])
    price_curve_ids = set(MARKET_ID_MAP["price_curve"])
    hazard_ids = set(MARKET_ID_MAP["hazard_curve"]) - discount_ids

    for curve_id in sorted(discount_ids):
        market.insert(DiscountCurve(curve_id, as_of, DISCOUNT_KNOTS, day_count="act_365f"))

    for curve_id in sorted(forward_ids):
        tenor = FORWARD_TENORS.get(curve_id, 0.25)
        market.insert(ForwardCurve(curve_id, tenor, FORWARD_KNOTS, base_date=as_of, day_count="act_365f"))

    for curve_id in sorted(price_curve_ids):
        level = PRICE_CURVE_LEVELS.get(curve_id, 100.0)
        market.insert(PriceCurve(curve_id, as_of, _price_curve_knots(level), day_count="act_365f"))

    for curve_id in sorted(hazard_ids):
        market.insert(
            HazardCurve(
                curve_id,
                as_of,
                [(0.25, 0.015), (1.0, 0.018), (3.0, 0.022), (5.0, 0.026), (10.0, 0.032)],
                recovery_rate=0.40,
                day_count="act_365f",
            )
        )

    for curve_id in sorted(MARKET_ID_MAP["inflation_curve"]):
        market.insert(
            InflationCurve(
                curve_id,
                as_of,
                315.0,
                [(0.0, 315.0), (1.0, 322.9), (2.0, 331.0), (5.0, 357.2), (10.0, 405.0)],
                day_count="act_365f",
            )
        )

    expiries = [0.01, 0.10, 0.25, 0.50, 1.0, 2.0, 5.0, 10.0]
    strikes = [0.0, 0.01, 0.03, 0.05, 0.08, 1.0, 20.0, 50.0, 75.0, 100.0, 150.0, 250.0, 4500.0, 5200.0, 6000.0, 40000.0]
    for surface_id in sorted(MARKET_ID_MAP["vol_surface"]):
        quote_type = "normal" if _is_rate_vol_surface(surface_id) else "black_lognormal"
        market.insert(
            VolSurface(
                surface_id,
                expiries,
                strikes,
                [0.25] * (len(expiries) * len(strikes)),
                secondary_axis="strike",
                interpolation_mode="vol",
                quote_type=quote_type,
            )
        )

    cube_expiries = [0.10, 0.25, 0.50, 1.0, 2.0, 5.0, 10.0]
    cube_tenors = [0.25, 1.0, 2.0, 5.0, 10.0, 30.0]
    cube_node_count = len(cube_expiries) * len(cube_tenors)
    cube_params = [{"alpha": 0.035, "beta": 0.50, "rho": -0.20, "nu": 0.40, "shift": 0.02}] * cube_node_count
    for cube_id in sorted(MARKET_ID_MAP["sabr_cube"]):
        market.insert(VolCube(cube_id, cube_expiries, cube_tenors, cube_params, [0.045] * cube_node_count))

    for curve_id in sorted(MARKET_ID_MAP["vol_index_curve"]):
        market.insert(VolatilityIndexCurve(curve_id, as_of, [(0.0, 18.0), (0.25, 19.0), (1.0, 20.0), (2.0, 21.0)]))

    fx = FxMatrix()
    fx.set_quote("EUR", "USD", 1.08)
    fx.set_quote("JPY", "USD", 0.0068)
    fx.set_quote("CNY", "USD", 0.1400)
    market.insert_fx(fx)

    for price_id in sorted(MARKET_ID_MAP["scalar_price"] | set(SCALAR_PRICES)):
        value = SCALAR_PRICES.get(price_id, 100.0)
        currency = None if price_id in {"CNYUSD-SPOT", "EURUSD", "EURUSD-SPOT", "JPYUSD-SPOT", "SOFR-RATE", "VIX", "US-CORP-CONVEXITY", "US-CORP-DURATION", "US-CORP-YIELD"} else "USD"
        market.insert_price(price_id, value, currency)
    for div_id in sorted(MARKET_ID_MAP["dividend_yield"] | set(DIVIDEND_YIELDS)):
        market.insert_price(div_id, DIVIDEND_YIELDS.get(div_id, 0.01), None)

    state = json.loads(market.to_json())
    state["fx"] = {
        "config": {"pivot_currency": "USD", "enable_triangulation": True, "cache_capacity": 256},
        "quotes": [["EUR", "USD", 1.08], ["JPY", "USD", 0.0068], ["CNY", "USD", 0.1400]],
    }

    # Some JSON-only pricers read scalar quotes from a compact price map.
    state.setdefault("prices", {})
    for price_id, value in SCALAR_PRICES.items():
        if price_id in {"CNYUSD-SPOT", "EURUSD", "EURUSD-SPOT", "JPYUSD-SPOT", "SOFR-RATE", "VIX", "US-CORP-CONVEXITY", "US-CORP-DURATION", "US-CORP-YIELD"}:
            state["prices"][price_id] = {"unitless": value}
        else:
            state["prices"][price_id] = {"price": {"amount": value, "currency": "USD"}}
    for div_id, value in DIVIDEND_YIELDS.items():
        state["prices"][div_id] = {"unitless": value}

    state["curves"].append(
        {
            "type": "base_correlation",
            "id": "CDX-BC",
            "detachment_points": [3.0, 7.0, 10.0, 15.0, 30.0, 100.0],
            "correlations": [0.30, 0.30, 0.30, 0.30, 0.30, 0.30],
        }
    )
    state["credit_indices"] = [
        {
            "id": "CDX.NA.IG.HAZARD",
            "num_constituents": 125,
            "recovery_rate": 0.40,
            "index_credit_curve_id": "CDX.NA.IG.HAZARD",
            "base_correlation_curve_id": "CDX-BC",
        }
    ]
    state["series"] = [
        _fixing_series("FIXING:USD-SOFR-3M", as_of),
        _fixing_series("FIXING:USD-SOFR-6M", as_of),
        _fixing_series("FIXING:SOFR-3M", as_of),
        _fixing_series("FIXING:SOFR-1M", as_of),
        _fixing_series("FIXING:NG-SPOT-AVG", as_of),
    ]
    return json.dumps(state)


SHARED_MARKET_JSON = build_shared_market_json()
SHARED_MARKET = json.loads(SHARED_MARKET_JSON)
print(
    "Shared market:",
    f"{len(SHARED_MARKET['curves'])} curves,",
    f"{len(SHARED_MARKET['surfaces'])} surfaces,",
    f"{len(SHARED_MARKET['vol_cubes'])} SABR cubes,",
    f"{len(SHARED_MARKET['prices'])} scalar prices",
)

Shared market: 27 curves, 19 surfaces, 6 SABR cubes, 24 scalar prices


In [4]:
from collections import Counter

from finstack_quant.valuations import ValuationResult

# Default models price the broad majority once the shared market is complete. Keep this map
# intentionally small; add entries only when a registered non-default model materially helps.
MODEL_OVERRIDES = {}

RESIDUAL_NOTES = {}


def _as_valuation_result(price_output):
    if isinstance(price_output, str):
        return ValuationResult.from_json(price_output)
    return price_output


def _short_error(exc):
    message = str(exc).split("\n")[0]
    return message.replace("Validation error: ", "")[:180]


def _failure_note(type_tag, exc):
    if type_tag in RESIDUAL_NOTES:
        return RESIDUAL_NOTES[type_tag]
    return "needs: " + _short_error(exc)


def _price_text(value):
    try:
        return f"{float(value):,.2f}"
    except (TypeError, ValueError):
        return str(value)


PRICE_ROWS = []
RESIDUAL_ROWS = []

for module_name, instruments in CATALOG.items():
    module = importlib.import_module(f"finstack_quant.valuations.instruments.{module_name}")
    for class_name, payload in instruments.items():
        cls = getattr(module, class_name)
        inst = cls.from_json(json.dumps(payload))
        inst.validate()
        type_tag = payload["type"]
        try:
            if type_tag in MODEL_OVERRIDES:
                price_output = inst.price(SHARED_MARKET_JSON, AS_OF.isoformat(), model=MODEL_OVERRIDES[type_tag])
            else:
                price_output = inst.price(SHARED_MARKET_JSON, AS_OF.isoformat())
            result = _as_valuation_result(price_output)
            PRICE_ROWS.append(
                {
                    "module": module_name,
                    "class": class_name,
                    "type": type_tag,
                    "price": getattr(result, "price", None),
                    "currency": str(getattr(result, "currency", "")),
                }
            )
        except TypeError as exc:
            # Credit-derivative wrappers expose a no-model signature; other wrappers accept the default keyword.
            try:
                price_output = inst.price(SHARED_MARKET_JSON, AS_OF.isoformat(), model="default")
                result = _as_valuation_result(price_output)
                PRICE_ROWS.append(
                    {
                        "module": module_name,
                        "class": class_name,
                        "type": type_tag,
                        "price": getattr(result, "price", None),
                        "currency": str(getattr(result, "currency", "")),
                    }
                )
            except Exception as fallback_exc:
                RESIDUAL_ROWS.append(
                    {
                        "module": module_name,
                        "class": class_name,
                        "type": type_tag,
                        "note": _failure_note(type_tag, fallback_exc),
                        "error": _short_error(fallback_exc),
                    }
                )
        except Exception as exc:
            RESIDUAL_ROWS.append(
                {
                    "module": module_name,
                    "class": class_name,
                    "type": type_tag,
                    "note": _failure_note(type_tag, exc),
                    "error": _short_error(exc),
                }
            )

module_totals = Counter()
for module_name, instruments in CATALOG.items():
    module_totals[module_name] = len(instruments)
module_priced = Counter(row["module"] for row in PRICE_ROWS)
module_residual = Counter(row["module"] for row in RESIDUAL_ROWS)

print(f"Priced {len(PRICE_ROWS)}/{sum(module_totals.values())} instruments against one shared MarketContext")
print("\nPer-module summary:")
for module_name in sorted(module_totals):
    print(
        f"  {module_name:18} priced {module_priced[module_name]:2d}/{module_totals[module_name]:2d}; "
        f"annotated {module_residual[module_name]:2d}"
    )

print("\nPriced PVs:")
for row in PRICE_ROWS:
    print(f"  {row['module']:18} {row['class']:28} {_price_text(row['price']):>14} {row['currency']}")

print("\nAnnotated residuals:")
for row in RESIDUAL_ROWS:
    print(f"  {row['module']:18} {row['class']:28} {row['note']} ({row['error']})")

Priced 70/70 instruments against one shared MarketContext

Per-module summary:
  commodity          priced  6/ 6; annotated  0
  credit_derivatives priced  4/ 4; annotated  0
  equity             priced 13/13; annotated  0
  exotics            priced  4/ 4; annotated  0
  fixed_income       priced 12/12; annotated  0
  fx                 priced 10/10; annotated  0
  rates              priced 21/21; annotated  0

Priced PVs:
  commodity          CommodityAsianOption               9,091.25 USD
  commodity          CommodityForward                       0.00 USD
  commodity          CommodityOption                    8,802.77 USD
  commodity          CommoditySpreadOption                  0.00 USD
  commodity          CommoditySwap                    -25,016.26 USD
  commodity          CommoditySwaption                 33,978.46 USD
  credit_derivatives CDSIndex                         399,031.29 USD
  credit_derivatives CDSOption                        384,959.05 USD
  credit_derivatives

## Takeaways

- All instrument classes share `from_json` / `validate` / `to_json` / `price`; the JSON discriminator `"type"` selects the class.
- The typed surface is the object-oriented counterpart to the JSON-first `price_instrument` / `price_instrument_with_metrics` functions used in the per-asset-class notebooks.
- Credit-derivative classes (`CDSIndex`, `CDSOption`, `CDSTranche`, `CreditDefaultSwap`) return a `ValuationResult` object directly from `price` (no `model=` argument).